# Study report: code-placement

The study asks how much the address of the code changes the latency, and
which placement gives numbers that a comparison between trees and
options can use. A pad (`LATBENCH_CODE_PAD_BYTES`, `common/src/pad.c`)
moves all app and kernel code by a number of bytes; nothing else
changes. The choice `LATBENCH_CODE_PLACE` selects the place of the code:
`LATBENCH_CODE_FLASH_PAD` (the code is in flash, with the pad) or
`LATBENCH_CODE_RAM` (the hot code is in the CCM SRAM, and the pad moves
only the relocated code). Code in the CCM SRAM does not go through the
flash and its cache.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-measurement-validity@nucleo_g474re/st-code-placement"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts"))
from probatrix.benchmarks import data  # noqa: E402
from probatrix.benchmarks import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The points

`wake-code-layout` has the code in flash, with the pads 0..28 bytes in
steps of 4 (`p0` .. `p28`), and 200000 samples per point.
`wake-code-place-q4` has hot code in RAM with the same pads (`r0` ..
`r28`) and a flash control (`f0`, `f4`: the code layout of `p0` and
`p4`), with 50000 samples per point. Both run on the tree `ws-up`, with
the anchors `default` (time slicing) and `noslice` (`TIMESLICING=n`, no
slice reprogram of SysTick). The series name gives the place and the
anchor. With `LATBENCH_CODE_RAM`, `code_pad_at` is the start of
`code_ram_text` (acceptance check A4). `drift/wake` is `clock_drift_cyc`
/ samples.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from probatrix.benchmarks.studylib import stats  # noqa: E402

In [ ]:
parts, metas = [], []
for e in ("wake-code-layout", "wake-code-place-q4"):
    if not nbfig.built(inst, e):
        continue
    df = inst.rows(e)
    if e == "wake-code-layout":
        df["pad"], df["place"] = df["value.pad"], "flash"
    else:
        df["pad"] = df["value.place"]
        df["place"] = np.where(df["axis.place"].str.startswith("r"), "RAM", "flash control")
    df["series"] = df.place + " " + df["axis.anchor"]
    df["experiment"] = e
    parts.append(df)
    pts = df.drop_duplicates("point")[["point", "experiment", "series", "place", "axis.anchor", "pad"]]
    m = inst.drift([e]).drop(columns="experiment").rename(columns={"drift_per_sample": "drift/wake"})
    metas.append(pts.merge(m, on="point"))
COLS = ["experiment", "point", "series", "place", "axis.anchor", "pad"]
rows = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=COLS + ["metric", "p50", "tree"])
points = pd.concat(metas, ignore_index=True) if metas else pd.DataFrame(
    columns=COLS + ["samples", "code_place", "code_pad", "code_pad_at", "clock_drift_cyc", "drift/wake"])
points[["experiment", "point", "series", "pad", "samples", "code_place", "code_pad", "code_pad_at", "clock_drift_cyc",
        "drift/wake"]]

## H1 a: the latency over the pad

H1 predicts that the latency and the drift change with the pad in flash,
with a period of the flash line or of the cache line. H2 predicts that
hot code in RAM removes the effect of the code address. The figure shows
the p50 over the pad for each series; the menu selects the metric. The
design gives no line size, so a person gives the verdict of H1 a.

In [ ]:
r = rows.assign(**{"value.pad (bytes)": rows.pad})
nbfig.distinct(nbfig.sweep(r, "value.pad (bytes)", "p50 over the pad: flash against RAM", stats=("p50",),
                     group="series")).show()
r[r.metric.isin(["irq", "isr2thr"])].pivot_table(index="pad", columns=["metric", "series"], values="p50",
                                                 aggfunc="first")

## H1 a and H2 c: the drift over the pad

The drift per wake for each series. With `noslice`, the anchor has no
slice reprogram of SysTick. The y axis is logarithmic when all values
are above 0.

In [ ]:
fig = go.Figure()
for k, s in enumerate(dict.fromkeys(points.series)):
    sub = points[points.series == s].sort_values("pad")
    fig.add_trace(go.Scatter(x=sub.pad, y=sub["drift/wake"], mode="lines+markers", name=s, customdata=sub.point,
                             line={"color": nbfig.color(k), "width": 2}, marker={"size": 8},
                             hovertemplate="pad %{x}: %{y:.3f}<br>%{customdata}<extra>" + s + "</extra>"))
positive = bool(len(points)) and (points["drift/wake"] > 0).all()
nbfig.style(nbfig.distinct(fig), "Cycles lost per wake over the pad", "pad (bytes)", "clock_drift_cyc / samples",
            yaxis_type="log" if positive else "linear").show()
points.pivot_table(index="pad", columns="series", values="drift/wake", aggfunc="first")

## H2 a..e: the spread over the pads in RAM

Each check of H2 takes the spread (max - min) of a value over the RAM
pads `r0` .. `r28` of one anchor. The grey bar is the limit of the
check. For comparison, the second series is the same spread over the
flash pads of `wake-code-layout`; no check reads it.

In [ ]:
CHECKS = [("H2 a", "irq", "p50", "default", 2), ("H2 b", "irq", "p50", "noslice", 2),
          ("H2 d", "isr2thr", "p50", "default", 5), ("H2 e", "isr2thr", "p50", "noslice", 5)]
out = []
for cid, metric, st, anchor, lim in CHECKS:
    for place, name in (("RAM", "RAM (wake-code-place-q4)"), ("flash", "flash (wake-code-layout)")):
        sub = rows[(rows.place == place) & (rows["axis.anchor"] == anchor) & (rows.metric == metric)]
        if len(sub):
            out.append({"check": f"{cid}: {metric} {st}, {anchor}", "series": name, "value": sub[st].max() - sub[st].min(),
                        "min": sub[st].min(), "max": sub[st].max(), "lo": 0, "hi": lim})
spread = pd.DataFrame(out)
if len(spread):
    nbfig.limits(spread, "Spread over the pads 0..28, against the limit of each check", "max - min (cycles)").show()
spread

In [ ]:
out = []
for place, name in (("RAM", "RAM (wake-code-place-q4)"), ("flash", "flash (wake-code-layout)")):
    sub = points[(points.place == place) & (points["axis.anchor"] == "default")]
    if len(sub):
        out.append({"check": "H2 c: drift/wake, default", "series": name,
                    "value": sub["drift/wake"].max() - sub["drift/wake"].min(),
                    "min": sub["drift/wake"].min(), "max": sub["drift/wake"].max(), "lo": 0, "hi": 0.2})
dspread = pd.DataFrame(out)
if len(dspread):
    nbfig.limits(dspread, "Spread of the drift over the pads 0..28", "max - min (cycles per wake)").update_layout(
        height=380).show()
dspread

## The cost of RAM (central result ram-cost)

The CCDF of the flash control (`f0`, `f4`) and of two RAM pads (`r0`,
`r28`), for both anchors (`noslice` dashed). The menu selects the
metric. The table gives the percentiles of all points of
`wake-code-place-q4`.

In [ ]:
SHOWN = ["f0-default", "f4-default", "r0-default", "r28-default", "f0-noslice", "f4-noslice", "r0-noslice", "r28-noslice"]
if nbfig.built(inst, "wake-code-place-q4"):
    h = nbfig.hists_of(inst, "wake-code-place-q4")
    nbfig.ccdf({p: h[p] for p in SHOWN if p in h}, "wake-code-place-q4: flash control against RAM",
               dashed={p for p in SHOWN if p.endswith("noslice")}).show()
    q4 = rows[rows.experiment == "wake-code-place-q4"]
    display(q4.pivot_table(index="point", columns="metric", values=["p50", "p99", "max"], aggfunc="first", sort=False))

## H2 k: a shift of the CCM code by 16 bytes

A new `LAT META` line in `stim_init()` shifts the CCM code by 16 bytes.
H2 k expects that the default seed of `wake-stim-seed-1m` (study
stimulus-sampling) gives the histograms of `baseline-t64` of
`wake-kconfig-1m` (study kconfig-factors), bin for bin. Both are
imports.

In [ ]:
if nbfig.built(inst, "wake-stim-seed-1m") and nbfig.built(inst, "wake-kconfig-1m"):
    a = next(rep for _, _, p, rep in inst.points("wake-stim-seed-1m") if p == "seed-default")
    b = next(rep for _, _, p, rep in inst.points("wake-kconfig-1m") if p == "baseline-t64")
    A, B = "wake-stim-seed-1m seed-default", "wake-kconfig-1m baseline-t64"
    nbfig.ccdf({A: a["hists"], B: b["hists"]}, "H2 k: the CCM code shift", ["irq", "isr2thr", "irq2thr"],
               dashed={B}).show()
    display(pd.DataFrame([{"metric": m, f"n ({A})": a["hists"][m]["n"], f"n ({B})": b["hists"][m]["n"],
                           f"max ({A})": a["hists"][m]["max"], f"max ({B})": b["hists"][m]["max"],
                           "the same, bin for bin": stats.hist_identical(a["hists"][m], b["hists"][m])}
                          for m in ("irq", "isr2thr", "irq2thr")]))
    display(pd.DataFrame([{"point": A, "code_pad_at": a["meta"].get("code_pad_at")},
                          {"point": B, "code_pad_at": b["meta"].get("code_pad_at")}]))

## H3: the place of the data

`wake-data-place-250k` moves the kernel data and the stacks with a data
pad (`LATBENCH_DATA_PAD`), with the code in the CCM SRAM.
`LAT META msp` is the top of the interrupt stack: SRAM2 starts at
`0x20014000`. `wake-witness-bss-250k` repeats the witness points of the
sub-study `tail-witnesses` at the revision whose witness table keeps the
stacks in SRAM1.

In [ ]:
for exp in ("wake-data-place-250k", "wake-witness-bss-250k"):
    if nbfig.built(inst, exp):
        df = inst.rows(exp)
        nbfig.dots(df, f"Wake latency over the data place ({exp})", stats=("min", "p50", "p99")).show()
        display(nbfig.pivot(df, ["min", "p50", "mean"]))
        m = inst.meta(exp)
        display(m[[k for k in ("point", "data_pad", "msp") if k in m.columns]])